In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

C = {
    'input':    '#E8F5E9',
    'conv':     '#BBDEFB',
    'pool':     '#FFF9C4',
    'fc':       '#FFE0B2',
    'output':   '#FFCDD2',
    'attention':'#E1BEE7',
    'fusion':   '#B2EBF2',
    'backbone': '#C8E6C9',
    'text':     '#2C3E50',
    'arrow':    '#546E7A',
}

In [ ]:
def draw_block(ax, x, y, w, h, label, sublabel='', color='#BBDEFB'):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                         facecolor=color, edgecolor='#37474F', linewidth=1.5, zorder=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color=C['text'], zorder=3)
    if sublabel:
        ax.text(x + w/2, y + h*0.25, sublabel, ha='center', va='center',
                fontsize=7, color='#546E7A', zorder=3)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=C['arrow'], lw=1.5), zorder=1)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.set_title('POSTER Architecture (horizontal flow)', fontsize=15, fontweight='bold', pad=15)

# --- Input ---
draw_block(ax, 0.5, 3.5, 1.8, 1.4, 'Input\n224x224x3', '', C['input'])

# --- IR-50 branch (upper) ---
ir_blocks = [
    (3.0, 5.0, 2.2, 1.3, 'IR-50 Backbone', 'SE blocks -> 512x7x7', C['backbone']),
    (5.8, 5.3, 1.5, 0.9, 'Reshape', '-> 49x1024', C['conv']),
    (8.0, 5.3, 1.5, 0.9, 'Linear', '1024->512', C['conv']),
]
for x, y, w, h, lbl, sub, c in ir_blocks:
    draw_block(ax, x, y, w, h, lbl, sub, c)

# Arrows within IR branch
draw_arrow(ax, 3.0-0.05, 5.0+0.65, 3.0, 5.0+0.65)
draw_arrow(ax, 3.0+2.2, 5.0+0.65, 5.8, 5.3+0.45)
draw_arrow(ax, 5.8+1.5, 5.3+0.45, 8.0, 5.3+0.45)

# --- MobileFaceNet branch (lower) ---
mfn_blocks = [
    (3.0, 1.3, 2.2, 1.3, 'MobileFaceNet', 'depthwise conv', C['backbone']),
    (5.8, 1.6, 1.5, 0.9, 'Reshape+Transpose', '-> 49x512', C['conv']),
]
for x, y, w, h, lbl, sub, c in mfn_blocks:
    draw_block(ax, x, y, w, h, lbl, sub, c)

draw_arrow(ax, 3.0-0.05, 1.3+0.65, 3.0, 1.3+0.65)
draw_arrow(ax, 3.0+2.2, 1.3+0.65, 5.8, 1.6+0.45)

# --- Arrows: Input -> both branches ---
draw_arrow(ax, 2.3, 3.5+0.7, 3.0, 5.0+0.65)
ax.text(2.65, 4.8, '224', fontsize=7, color='#546E7A', style='italic')
draw_arrow(ax, 2.3, 3.5+0.7, 3.0, 1.3+0.65)
ax.text(2.65, 3.0, '112', fontsize=7, color='#546E7A', style='italic')

# --- Concat ---
draw_arrow(ax, 8.0+1.5, 5.3+0.45, 10.5, 5.5+0.6)
draw_arrow(ax, 5.8+1.5, 1.6+0.45, 10.5, 5.5+0.6)
draw_block(ax, 10.0, 5.0, 1.0, 1.2, 'Concat', '', C['pool'])

# --- Fusion & Head (rightward) ---
draw_arrow(ax, 11.0, 5.0+0.6, 12.5, 3.5+0.35)

draw_block(ax, 12.5, 2.8, 2.5, 1.4, 'HyVisionTransformer\n(Pyramid Fusion)',
           'cross-attention\n8 heads x 8 depth', C['fusion'])
draw_arrow(ax, 15.0, 2.8+0.7, 16.0, 2.0+0.5)

draw_block(ax, 16.0, 1.5, 1.8, 1.0, 'SE Block', 'channel attn', C['attention'])
draw_arrow(ax, 17.8, 1.5+0.5, 18.8, 1.5+0.5)

draw_block(ax, 18.8, 1.5, 1.2, 1.0, 'Dropout', '0.3', C['pool'])
draw_arrow(ax, 20.0, 1.5+0.5, 21.0, 1.5+0.5)

draw_block(ax, 21.0, 1.2, 1.8, 1.4, 'Linear Head\n512->7', 'softmax', C['output'])

ax.set_xlim(0, 23.5)
ax.set_ylim(0, 7.5)
ax.axis('off')

plt.tight_layout()
plt.savefig('poster_architecture.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: poster_architecture.png')